In [ ]:
# Jupyter notebook for Session 0.
# NB! Restart kernel and clear all outputs before committing to git.

from dotenv import load_dotenv
import os
import sqlalchemy as sa
import pandas as pd
from supabase import create_client

In [ ]:
# Load configuration from .env file in parent directory.
# dotenv is smart enough to search whole directory tree for .env files.
load_dotenv(override=True)

# Connect to Supabase directly.
supabase_direct = sa.create_engine(os.getenv("SUPABASE_CONNECTION_STRING"))

# And via API, for demonstration purposes.
supabase_api = create_client(os.getenv("SUPABASE_URL"), os.getenv("SUPABASE_KEY"))

# Loading data using SQL query over direct connection.
# More flexible control over sizes of data sets.
def load_data_using_sql(sql):
    query = sa.text(sql)
    return pd.read_sql(query, supabase_direct)

# Loading data over supabase API.
# Should not be done in real-world situations when working with tables with millions of rows!
def load_data_using_api(table):
    response = supabase_api.table(table).select('*').execute()
    return pd.DataFrame(response.data)


In [ ]:
# Load all data from sales table.
df = load_data_using_api("sales")
df

In [ ]:
# 5 first rows from sales table.
df.head()

In [ ]:
# number of rows x number of columns
df.shape

In [ ]:
# Number of rows in sales table.
df.shape[0]

In [ ]:
# Generate descriptive statistics.
df.describe()

In [ ]:
# Columns, data types, count of NOT NULL values.
df.info()

In [ ]:
# Let's define a dataset.
data = {
    'customer_id': [1001, 1002, 1003, 1001, 1002, 1004, 1003, 1001, 1005, 1004,
                    1002, 1003, 1005, 1001, 1006, 1004, 1002, 1007, 1003, 1005],
    'sale_date': ['2024-01-15', '2024-01-16', '2024-02-01', '2024-02-20', '2024-03-01',
                   '2024-03-05', '2024-03-15', '2024-04-10', '2024-04-12', '2024-04-20',
                   '2024-05-01', '2024-05-10', '2024-05-15', '2024-06-01', '2024-06-05',
                   '2024-06-10', '2024-06-20', '2024-07-01', '2024-07-05', '2024-07-10'],
    'total_price': [89.99, 45.50, 120.00, 67.30, 55.00, 210.00, 33.50, 145.00, 78.00, 92.00,
                     160.00, 44.00, 88.50, 230.00, 37.00, 175.00, 110.00, 65.00, 95.00, 125.00],
    'city': ['Tallinn', 'Tartu', 'Tallinn', 'Tallinn', 'Tartu', 'Pärnu', 'Tallinn', 'Tallinn',
             'Tartu', 'Pärnu', 'Tartu', 'Tallinn', 'Tartu', 'Tallinn', 'Pärnu', 'Pärnu',
             'Tartu', 'Tallinn', 'Tallinn', 'Tartu'],
    'product_category': ['Dresses', 'Tops', 'Denim', 'Accessories', 'Tops', 'Denim', 'Tops',
                         'Dresses', 'Denim', 'Accessories', 'Dresses', 'Tops', 'Denim',
                         'Dresses', 'Accessories', 'Denim', 'Tops', 'Accessories', 'Dresses', 'Denim']
}

# Constructing a pandas DataFrame using the given data set.
df = pd.DataFrame(data)

In [ ]:
# Number of rows and columns.
df.shape

In [ ]:
# 5 first rows
df.head()

In [ ]:
# Basic info.
df.info()

In [ ]:
# Statistics.
df.describe()

In [ ]:
# Data types of columns.
df.dtypes

In [ ]:
# Number of unique customer ids.
df["customer_id"].nunique()

In [ ]:
# Unique city names in the data frame.
df["city"].unique()

In [ ]:
# Number of occurrences of each city in the data frame.
df["city"].value_counts()

In [ ]:
# Total revenue - sum of total price of each sale.
df["total_price"].sum()

In [ ]:
# Number of unique product categories. Similar to count(*) in PostgreSQL.
df["product_category"].nunique()

In [ ]:
# Unique product categories. Similar to SELECT DISTINCT in PostgreSQL.
df["product_category"].unique()

In [ ]:
# Count of values of each unique product category.
# Similar to: SELECT product_category, count(*) AS products_count FROM products GROUP BY product_category;
df["product_category"].value_counts()

In [ ]:
# Descriptive statistics for product_category column.
df["product_category"].describe()

In [ ]:
# Boolean indexing.
df["total_price"] > 100

In [ ]:
# Using boolean indexing to filter the data frame to include only rows where total_price is greater than 100.
# In SQL, it would be done like this: SELECT * FROM sales WHERE total_price > 100;
df[df["total_price"] > 100]

In [ ]:
# Filtered data frame where city is Tallinn in each row.
# SQL filter: WHERE city = 'Tallinn'
df[df["city"] == "Tallinn"]

In [ ]:
# Filtering pandas DataFrame by total price and city.
# SQL filter: WHERE total_price > 100 AND city = 'Tallinn'.
df[(df["total_price"] > 100) & (df["city"] == "Tallinn")]

In [ ]:
# Summarizing total revenue for each city.
# In SQL: use GROUP BY.
grouped = df.groupby("city")["total_price"].sum()
print(type(grouped))
grouped

In [ ]:
# Using multiple aggregate functions over a column.
grouped = df.groupby('city')['total_price'].agg(['sum', 'mean', 'count'])
print(type(grouped))
grouped

In [ ]:
# Note that agg(['sum']) behaves differently from sum(). It returns a DataFrame object instead of a Series.
grouped = df.groupby('city')['total_price'].agg(['sum'])
print(type(grouped))
grouped

In [ ]:
# Summarizing over multiple columns.
df.groupby(["city", "product_category"])["total_price"].sum()

In [ ]:
# Multiple aggregations over multiple columns.
grouped = df.groupby(["city", "product_category"])["total_price"].agg(["sum", "mean", "count", "min", "max"])

In [ ]:
# Sorting by total price in descending order.
# SQL: ORDER BY total_price DESC;
df.sort_values('total_price', ascending=False)

In [ ]:
# Sorting by multiple columns.
# SQL: ORDER BY city ASC, total_price DESC.
df.sort_values(['city', 'total_price'], ascending=[True, False])

In [ ]:
df_sales = load_data_using_sql("SELECT * FROM sales;")
df_customers = load_data_using_sql("SELECT * FROM customers;")

In [ ]:
# Joining pandas data frames.
# SQL:
"""
SELECT
    *
FROM sales s
LEFT JOIN customers c ON s.customer_id = s.customer_id
"""
merged = pd.merge(df_sales, df_customers, on='customer_id', how='left')
print(type(merged))
merged.head()

In [ ]:
# Let's check shapes.
# Number of rows in merged should be the same as df_sales.
# Number of columns in merged should be one less than the number of columns in df_sales plus df_customers,
# because customer_id column will not be duplicated in merged dataframe.
df_sales.shape, df_customers.shape, merged.shape

In [ ]:
# Adding a new column to pandas data frame.
df['discount'] = df['total_price'] * 0.1
df.head()

In [ ]:
# Conditional calculations on total_price column to add another column (segment).
# SQL: CASE WHEN.
df['segment'] = df['total_price'].apply(
    lambda x: 'Big' if x > 100 else 'Small'
)
df.head()

In [ ]:
# Converting sale dates to datetime.
df['sale_date'] = pd.to_datetime(df['sale_date'])
df.head()

In [ ]:
# Extracting month and year from sale date and storing them in extra columns.
df["month"] = df["sale_date"].dt.month
df["year"] = df["sale_date"].dt.year
df.head()

In [ ]:
df_sales.head()

In [ ]:
# Adding a new column to merged dataframe which adds business interpretation to total price.
merged["order_size"] = merged["total_price"].apply(
    lambda x: "Suur (100+)" if x >= 100 else "Väike (<100)"
)
merged.head()

In [ ]:
# All sales rows where customer city is Tallinn.
tallinn = merged[merged['city'] == 'Tallinn']
tallinn.describe()

In [ ]:
# Aggregations over all cities with cities with largest sum appearing at the top.
city_revenue = merged.groupby('city')['total_price'].agg(['sum', 'mean', 'count']).sort_values("sum", ascending=False)
# Rename columns to provide better business context:
city_revenue.columns = ["total_revenue", "average_revenue", "orders"]
city_revenue.head()

In [ ]:
# Statistics on aggregations (which is statistics by itself)
city_revenue.describe()

In [ ]:
order_sizes_series = merged['order_size'].value_counts()
print(type(order_sizes_series))

# reset_index() will convert Series to DataFrame:
order_sizes = order_sizes_series.reset_index()
print(type(order_sizes))

order_sizes

In [ ]:
# Top customer cities by average revenue.
city_revenue.sort_values("average_revenue", ascending=False).head(10)

In [ ]:
# Named Aggregation - naming aggregations in a different way than using `by_customer.columns =`
customer_summary = merged.groupby(["customer_id", "first_name", "last_name"]).agg(
    total_spending=("total_price", "sum"),
    average_order=("total_price", "mean"),
    orders=("total_price", "count")
).sort_values("total_spending", ascending=False).reset_index()

# Adding VIP status.
customer_summary["vip_status"] = customer_summary["total_spending"].apply(
    lambda x: "YES" if x > 200 else "NO"    # Täida lüngad!
)

# TOP 5 customers with id and name, by total spending.
customer_summary.head()

In [ ]:
# Find out how many VIP customers there are.
customer_summary["vip_status"].value_counts().reset_index()